# 承認済み部位別 before / after を解析する

`video_region_pair_candidates.ipynb` で固定した `selected_region_pairs.json` を使います。
現在選択されている部位だけを解析し、別部位のペアへ自動で置き換えません。


In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
marker = Path('analysis/analyze_selected_regions.py')
if (cwd / marker).is_file():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('repo root:', Path.cwd())


repo root: C:\Users\mail\work\ikiikimake


In [2]:
VIDEO = Path('makeup0923.mp4')
OUTPUT = Path('outputs') / VIDEO.stem
SELECTED = OUTPUT / 'selected_region_pairs.json'
ANALYSIS_ROOT = OUTPUT / 'selected_region_analysis'

if not SELECTED.is_file():
    raise FileNotFoundError(f'部位別ペアが固定されていません: {SELECTED}')
print('selected:', SELECTED)


selected: outputs\makeup0923\selected_region_pairs.json


In [ ]:
from analysis.analyze_selected_regions import analyze_selected_regions
from IPython.display import HTML, display
import base64

summary = analyze_selected_regions(SELECTED, ANALYSIS_ROOT)
RUN_DIR = Path(summary['output_dir'])
print('analysis:', RUN_DIR)

# report.html 内の相対画像パスは VS Code/Jupyter の HTML 表示では
# notebook 側を基準に解決されて壊れるため、表示時だけ data URI に埋め込む。
report_html = (RUN_DIR / 'report.html').read_text(encoding='utf-8')
for region in summary['regions']:
    for phase in ('before', 'after'):
        filename = f'{region}_{phase}.png'
        image_path = RUN_DIR / filename
        if not image_path.is_file():
            raise FileNotFoundError(image_path)
        encoded = base64.b64encode(image_path.read_bytes()).decode('ascii')
        report_html = report_html.replace(
            f"src='{filename}'",
            f"src='data:image/png;base64,{encoded}'",
        )
display(HTML(report_html))


In [4]:
for region, result in summary['regions'].items():
    print('\n', region)
    for row in result['deltas']:
        before = row['before']
        after = row['after']
        delta = row['delta']
        print(f"{row['label']}: {before:.4f} -> {after:.4f}  delta={delta:+.4f}")
print('\nreport:', RUN_DIR / 'report.html')
print('csv   :', RUN_DIR / 'feature_deltas.csv')



 eye_texture
画面左目の下の細かな質感コントラスト（中央値）: 0.3350 -> 1.6342  delta=+1.2992
画面左目の下の細かな質感コントラスト（p90）: 0.8295 -> 7.0019  delta=+6.1725
画面右目の下の細かな質感コントラスト（中央値）: 0.1843 -> 1.4917  delta=+1.3074
画面右目の下の細かな質感コントラスト（p90）: 0.5513 -> 8.1519  delta=+7.6005

report: C:\Users\mail\work\ikiikimake\outputs\makeup0923\selected_region_analysis\50234fe53483a3c6\report.html
csv   : C:\Users\mail\work\ikiikimake\outputs\makeup0923\selected_region_analysis\50234fe53483a3c6\feature_deltas.csv
